In [ ]:
# Cell 1 — Clone repos and install Python deps
!git clone https://github.com/ggml-org/whisper.cpp /content/whisper.cpp
!git clone https://github.com/openai/whisper /content/openai-whisper
!pip -q install -U torch transformers sentencepiece


Cloning into '/content/whisper.cpp'...
remote: Enumerating objects: 34814, done.
remote: Counting objects: 100% (1709/1709), done.
remote: Compressing objects: 100% (572/572), done.
remote: Total 34814 (delta 1158), reused 1453 (delta 1132), pack-reused 33105 (from 3)
Receiving objects: 100% (34814/34814), 35.70 MiB | 20.51 MiB/s, done.
Resolving deltas: 100% (25473/25473), done.
Cloning into '/content/openai-whisper'...
remote: Enumerating objects: 900, done.
remote: Total 900 (delta 0), reused 0 (delta 0), pack-reused 900 (from 1)
Receiving objects: 100% (900/900), 12.49 MiB | 30.10 MiB/s, done.
Resolving deltas: 100% (520/520), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 Why your WER is so high

From your setup:

Dataset size: ~3000 samples → too small
Whisper tiny → very low capacity
Malayalam ASR → harder task
No strong augmentation / normalization

Nothing is “wrong” — this is expected.

In [ ]:
# Cell 2 — Mount Drive and set paths
from google.colab import drive
drive.mount("/content/drive")

import os

HF_MODEL_DIR = "/content/drive/MyDrive/malayalam_whisper_tiny/final"
OUT_DIR      = "/content/drive/MyDrive/malayalam_whisper_tiny/ggml"
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.isdir(HF_MODEL_DIR), f"Model folder not found: {HF_MODEL_DIR}"
print("HF model:", HF_MODEL_DIR)
print("Output dir:", OUT_DIR)


Mounted at /content/drive
HF model: /content/drive/MyDrive/malayalam_whisper_tiny/final
Output dir: /content/drive/MyDrive/malayalam_whisper_tiny/ggml


In [ ]:
from transformers import WhisperProcessor

save_path = "/content/drive/MyDrive/malayalam_whisper_tiny/final"

base = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Copy tokenizer files into your trained model folder
base.tokenizer.save_pretrained(save_path)

print("✅ vocab.json + merges.txt copied")


✅ vocab.json + merges.txt copied


In [ ]:
from transformers import WhisperProcessor

proc = WhisperProcessor.from_pretrained("openai/whisper-tiny")

print("Downloaded tokenizer to cache")


Downloaded tokenizer to cache


In [ ]:
!ls ~/.cache/huggingface/hub/models--openai--whisper-tiny/snapshots/*/


added_tokens.json  normalizer.json	     tokenizer_config.json
config.json	   preprocessor_config.json  tokenizer.json
merges.txt	   special_tokens_map.json   vocab.json


In [ ]:
!cp ~/.cache/huggingface/hub/models--openai--whisper-tiny/snapshots/*/vocab.json \
    /content/drive/MyDrive/malayalam_whisper_tiny/final/

!cp ~/.cache/huggingface/hub/models--openai--whisper-tiny/snapshots/*/merges.txt \
    /content/drive/MyDrive/malayalam_whisper_tiny/final/


In [ ]:
!cp ~/.cache/huggingface/hub/models--openai--whisper-tiny/snapshots/*/added_tokens.json \
    /content/drive/MyDrive/malayalam_whisper_tiny/final/


In [ ]:
!ls /content/drive/MyDrive/malayalam_whisper_tiny/final


added_tokens.json	model.safetensors      training_args.bin
config.json		processor_config.json  vocab.json
generation_config.json	tokenizer_config.json
merges.txt		tokenizer.json


In [ ]:
import subprocess

result = subprocess.run([
    "python3",
    "/content/whisper.cpp/models/convert-h5-to-ggml.py",
    HF_MODEL_DIR,            # ← changed
    "/content/openai-whisper",
    OUT_DIR,                 # ← changed
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("✅ GGML conversion done")


model.encoder.conv1.weight  ->  encoder.conv1.weight
encoder.conv1.weight 3 (384, 80, 3)
model.encoder.conv1.bias  ->  encoder.conv1.bias
  Reshaped variable:  encoder.conv1.bias  to shape:  (384, 1)
encoder.conv1.bias 2 (384, 1)
  Converting to float32
model.encoder.conv2.weight  ->  encoder.conv2.weight
encoder.conv2.weight 3 (384, 384, 3)
model.encoder.conv2.bias  ->  encoder.conv2.bias
  Reshaped variable:  encoder.conv2.bias  to shape:  (384, 1)
encoder.conv2.bias 2 (384, 1)
  Converting to float32
model.encoder.embed_positions.weight  ->  encoder.positional_embedding
encoder.positional_embedding 2 (1500, 384)
  Converting to float32
model.encoder.layers.0.self_attn.k_proj.weight  ->  encoder.blocks.0.attn.key.weight
encoder.blocks.0.attn.key.weight 2 (384, 384)
model.encoder.layers.0.self_attn.v_proj.weight  ->  encoder.blocks.0.attn.value.weight
encoder.blocks.0.attn.value.weight 2 (384, 384)
model.encoder.layers.0.self_attn.v_proj.bias  ->  encoder.blocks.0.attn.value.bias
enco

In [ ]:
!rm -rf /content/whisper.cpp
!git clone https://github.com/ggerganov/whisper.cpp /content/whisper.cpp


Cloning into '/content/whisper.cpp'...
remote: Enumerating objects: 34814, done.
remote: Counting objects: 100% (1709/1709), done.
remote: Compressing objects: 100% (572/572), done.
remote: Total 34814 (delta 1158), reused 1453 (delta 1132), pack-reused 33105 (from 3)
Receiving objects: 100% (34814/34814), 35.70 MiB | 20.48 MiB/s, done.
Resolving deltas: 100% (25473/25473), done.


In [ ]:
!cd /content/whisper.cpp && make -j4


cmake -B build 
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assemb

In [ ]:
!ls /content/whisper.cpp


AUTHORS		      close-issue.yml  grammars  README.md	 tests
bindings	      cmake	       include	 README_sycl.md
build		      CMakeLists.txt   LICENSE	 samples
build-xcframework.sh  examples	       Makefile  scripts
ci		      ggml	       models	 src


In [ ]:
!cd /content/whisper.cpp && make -j4


cmake -B build 
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.8
-- ggml commit:  fc674574
-- Configuring done (0.3s)
-- Generating done (0.1s)
-- Build files have been written to: /content/whisper.cpp/build
cmake --build build --config Release
gmake[1]: warning: jobserver unavailable: using -j1.  Add '+' to parent make rule.
gmake[1]: Entering directory '/content/whisper.cpp/

In [ ]:
!cd /content/whisper.cpp && ./build/bin/whisper-cli \
-m "/content/drive/MyDrive/malayalam_whisper_tiny/ggml/ggml-model.bin" \
-l ml \
-f /content/nellu1.wav


whisper_init_from_file_with_params_no_state: loading model from '/content/drive/MyDrive/malayalam_whisper_tiny/ggml/ggml-model.bin'
whisper_init_with_params_no_state: use gpu    = 1
whisper_init_with_params_no_state: flash attn = 1
whisper_init_with_params_no_state: gpu_device = 0
whisper_init_with_params_no_state: dtw        = 0
whisper_init_with_params_no_state: devices    = 1
whisper_init_with_params_no_state: backends   = 1
whisper_model_load: loading model
whisper_model_load: n_vocab       = 51865
whisper_model_load: n_audio_ctx   = 1500
whisper_model_load: n_audio_state = 384
whisper_model_load: n_audio_head  = 6
whisper_model_load: n_audio_layer = 4
whisper_model_load: n_text_ctx    = 448
whisper_model_load: n_text_state  = 384
whisper_model_load: n_text_head   = 6
whisper_model_load: n_text_layer  = 4
whisper_model_load: n_mels        = 80
whisper_model_load: ftype         = 1
whisper_model_load: qntvr         = 0
whisper_model_load: type          = 1 (tiny)
whisper_model_load:

In [ ]:
!mkdir -p /content/drive/MyDrive/malayalam_whisper_tiny_backup
!cp -r /content/drive/MyDrive/malayalam_whisper_tiny/final /content/drive/MyDrive/malayalam_whisper_tiny_backup/
!cp -r /content/drive/MyDrive/malayalam_whisper_tiny/ggml /content/drive/MyDrive/malayalam_whisper_tiny_backup/
!zip -r /content/drive/MyDrive/malayalam_whisper_tiny_backup.zip /content/drive/MyDrive/malayalam_whisper_tiny_backup


  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/ (stored 0%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/ (stored 0%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/processor_config.json (deflated 48%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/config.json (deflated 60%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/tokenizer_config.json (deflated 74%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/generation_config.json (deflated 70%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/tokenizer.json (deflated 82%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/training_args.bin (deflated 53%)
  adding: content/drive/MyDrive/malayalam_whisper_tiny_backup/final/vocab.json (deflated 58%)
  adding: content/drive/MyDrive/malayala

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/malayalam_whisper_tiny_backup.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>